In [14]:
import pandas as pd
import sqlite3

# Load the Excel file
df = pd.read_excel('Dataset for Data Analytics (1).xlsx')

# Create SQLite database in memory
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

print("=" * 80)
print("E-COMMERCE DATA ANALYTICS PROJECT")
print("=" * 80)

# 1. BASIC STATISTICS
print("\n1. BASIC STATISTICS")
print("-" * 50)
query1 = """
SELECT 
    COUNT(*) as total_orders,
    ROUND(SUM(TotalPrice), 2) as total_revenue,
    ROUND(AVG(TotalPrice), 2) as avg_order_value,
    SUM(Quantity) as total_units_sold,
    COUNT(DISTINCT CustomerID) as unique_customers,
    COUNT(DISTINCT Product) as unique_products
FROM orders
"""
result1 = pd.read_sql(query1, conn)
print(result1.to_string(index=False))

# 2. PRODUCT PERFORMANCE
print("\n2. PRODUCT PERFORMANCE (Top 10)")
print("-" * 50)
query2 = """
SELECT 
    Product,
    COUNT(*) as orders,
    SUM(Quantity) as units_sold,
    ROUND(SUM(TotalPrice), 2) as revenue,
    ROUND(AVG(TotalPrice), 2) as avg_order_value
FROM orders
GROUP BY Product
ORDER BY revenue DESC
LIMIT 10
"""
result2 = pd.read_sql(query2, conn)
print(result2.to_string(index=False))

# 3. ORDER STATUS DISTRIBUTION
print("\n3. ORDER STATUS DISTRIBUTION")
print("-" * 50)
query3 = """
SELECT 
    OrderStatus,
    COUNT(*) as order_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) as percentage,
    ROUND(SUM(TotalPrice), 2) as total_value
FROM orders
GROUP BY OrderStatus
ORDER BY order_count DESC
"""
result3 = pd.read_sql(query3, conn)
print(result3.to_string(index=False))

# 4. PAYMENT METHOD ANALYSIS
print("\n4. PAYMENT METHOD ANALYSIS")
print("-" * 50)
query4 = """
SELECT 
    PaymentMethod,
    COUNT(*) as transaction_count,
    ROUND(SUM(TotalPrice), 2) as total_revenue,
    ROUND(AVG(TotalPrice), 2) as avg_order_value,
    ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) as revenue_percentage
FROM orders
GROUP BY PaymentMethod
ORDER BY total_revenue DESC
"""
result4 = pd.read_sql(query4, conn)
print(result4.to_string(index=False))

# 5. TOP 10 CUSTOMERS
print("\n5. TOP 10 CUSTOMERS BY SPENDING")
print("-" * 50)
query5 = """
SELECT 
    CustomerID,
    COUNT(*) as order_count,
    ROUND(SUM(TotalPrice), 2) as total_spent,
    ROUND(AVG(TotalPrice), 2) as avg_order_value
FROM orders
GROUP BY CustomerID
ORDER BY total_spent DESC
LIMIT 10
"""
result5 = pd.read_sql(query5, conn)
print(result5.to_string(index=False))

# 6. MARKETING CHANNEL PERFORMANCE
print("\n6. MARKETING CHANNEL / REFERRAL SOURCE PERFORMANCE")
print("-" * 50)
query6 = """
SELECT 
    ReferralSource,
    COUNT(*) as order_count,
    ROUND(SUM(TotalPrice), 2) as total_revenue,
    ROUND(AVG(TotalPrice), 2) as avg_order_value,
    ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) as revenue_share
FROM orders
GROUP BY ReferralSource
ORDER BY total_revenue DESC
"""
result6 = pd.read_sql(query6, conn)
print(result6.to_string(index=False))

# 7. COUPON PERFORMANCE
print("\n7. COUPON CODE PERFORMANCE")
print("-" * 50)
query7 = """
SELECT 
    CASE 
        WHEN CouponCode IS NULL OR CouponCode = '' THEN 'No Coupon'
        ELSE CouponCode 
    END as coupon_type,
    COUNT(*) as usage_count,
    ROUND(SUM(TotalPrice), 2) as total_revenue,
    ROUND(AVG(TotalPrice), 2) as avg_order_value
FROM orders
GROUP BY coupon_type
ORDER BY total_revenue DESC
"""
result7 = pd.read_sql(query7, conn)
print(result7.to_string(index=False))

# 8. HIGH VALUE ORDERS (> $2000)
print("\n8. HIGH VALUE ORDERS (> $2000)")
print("-" * 50)
query8 = """
SELECT 
    OrderID,
    CustomerID,
    Product,
    Quantity,
    ROUND(TotalPrice, 2) as order_value,
    OrderStatus,
    PaymentMethod,
    ReferralSource
FROM orders
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC
LIMIT 10
"""
result8 = pd.read_sql(query8, conn)
print(result8.to_string(index=False))

# 9. CANCELLED & RETURNED ORDERS ANALYSIS
print("\n9. CANCELLED & RETURNED ORDERS ANALYSIS")
print("-" * 50)
query9 = """
SELECT 
    OrderStatus,
    COUNT(*) as order_count,
    ROUND(SUM(TotalPrice), 2) as lost_revenue,
    ROUND(AVG(TotalPrice), 2) as avg_lost_value,
    COUNT(DISTINCT CustomerID) as unique_customers_affected
FROM orders
WHERE OrderStatus IN ('Cancelled', 'Returned')
GROUP BY OrderStatus
"""
result9 = pd.read_sql(query9, conn)
print(result9.to_string(index=False))

# 10. MONTHLY SALES TREND
print("\n10. MONTHLY SALES TREND (Last 12 Months)")
print("-" * 50)
query10 = """
SELECT 
    SUBSTR(Date, 1, 7) as year_month,
    COUNT(*) as order_count,
    SUM(Quantity) as units_sold,
    ROUND(SUM(TotalPrice), 2) as monthly_revenue,
    ROUND(AVG(TotalPrice), 2) as avg_order_value
FROM orders
GROUP BY year_month
ORDER BY year_month DESC
LIMIT 12
"""
result10 = pd.read_sql(query10, conn)
print(result10.to_string(index=False))

# 11. PRODUCT DEMAND BY QUANTITY
print("\n11. MOST DEMANDED PRODUCTS (By Units Sold)")
print("-" * 50)
query11 = """
SELECT 
    Product,
    SUM(Quantity) as total_demand,
    COUNT(*) as orders,
    ROUND(AVG(UnitPrice), 2) as avg_price,
    ROUND(SUM(TotalPrice), 2) as revenue
FROM orders
GROUP BY Product
ORDER BY total_demand DESC
LIMIT 10
"""
result11 = pd.read_sql(query11, conn)
print(result11.to_string(index=False))

# 12. AVERAGE ITEMS IN CART BY PRODUCT
print("\n12. AVERAGE ITEMS IN CART BY PRODUCT")
print("-" * 50)
query12 = """
SELECT 
    Product,
    ROUND(AVG(ItemsInCart), 2) as avg_items_in_cart,
    MAX(ItemsInCart) as max_items_in_cart,
    COUNT(*) as orders,
    ROUND(SUM(TotalPrice), 2) as revenue
FROM orders
GROUP BY Product
ORDER BY avg_items_in_cart DESC
LIMIT 10
"""
result12 = pd.read_sql(query12, conn)
print(result12.to_string(index=False))

# 13. ORDER STATUS BY PAYMENT METHOD
print("\n13. ORDER STATUS BREAKDOWN BY PAYMENT METHOD")
print("-" * 50)
query13 = """
SELECT 
    PaymentMethod,
    OrderStatus,
    COUNT(*) as count,
    ROUND(SUM(TotalPrice), 2) as total_value
FROM orders
GROUP BY PaymentMethod, OrderStatus
ORDER BY PaymentMethod, count DESC
"""
result13 = pd.read_sql(query13, conn)
print(result13.to_string(index=False))

# 14. PRODUCT PERFORMANCE BY ORDER STATUS
print("\n14. TOP PRODUCTS WITH HIGHEST CANCELLATION/RETURN RATE")
print("-" * 50)
query14 = """
SELECT 
    Product,
    COUNT(*) as total_orders,
    SUM(CASE WHEN OrderStatus IN ('Cancelled', 'Returned') THEN 1 ELSE 0 END) as failed_orders,
    ROUND(100.0 * SUM(CASE WHEN OrderStatus IN ('Cancelled', 'Returned') THEN 1 ELSE 0 END) / COUNT(*), 2) as failure_rate,
    ROUND(SUM(CASE WHEN OrderStatus IN ('Cancelled', 'Returned') THEN TotalPrice ELSE 0 END), 2) as lost_revenue
FROM orders
GROUP BY Product
HAVING failed_orders > 0
ORDER BY failure_rate DESC
LIMIT 10
"""
result14 = pd.read_sql(query14, conn)
print(result14.to_string(index=False))

# 15. EXECUTIVE SUMMARY
print("\n15. EXECUTIVE SUMMARY DASHBOARD")
print("=" * 50)
query15 = """
SELECT 'Total Orders' as Metric, CAST(COUNT(*) AS VARCHAR) as Value FROM orders
UNION ALL
SELECT 'Total Revenue', CAST(ROUND(SUM(TotalPrice), 2) AS VARCHAR) FROM orders
UNION ALL
SELECT 'Average Order Value', CAST(ROUND(AVG(TotalPrice), 2) AS VARCHAR) FROM orders
UNION ALL
SELECT 'Total Units Sold', CAST(SUM(Quantity) AS VARCHAR) FROM orders
UNION ALL
SELECT 'Unique Customers', CAST(COUNT(DISTINCT CustomerID) AS VARCHAR) FROM orders
UNION ALL
SELECT 'Unique Products', CAST(COUNT(DISTINCT Product) AS VARCHAR) FROM orders
UNION ALL
SELECT 'Successful Orders (Shipped/Delivered)', CAST(COUNT(*) AS VARCHAR) FROM orders WHERE OrderStatus IN ('Shipped', 'Delivered')
UNION ALL
SELECT 'Failed Orders (Cancelled/Returned)', CAST(COUNT(*) AS VARCHAR) FROM orders WHERE OrderStatus IN ('Cancelled', 'Returned')
UNION ALL
SELECT 'Pending Orders', CAST(COUNT(*) AS VARCHAR) FROM orders WHERE OrderStatus = 'Pending'
UNION ALL
SELECT 'Order Success Rate', CAST(ROUND(100.0 * (SELECT COUNT(*) FROM orders WHERE OrderStatus IN ('Shipped', 'Delivered')) / COUNT(*), 2) AS VARCHAR) || '%' FROM orders
"""
result15 = pd.read_sql(query15, conn)
print(result15.to_string(index=False))

# Close connection
conn.close()

print("\n" + "=" * 80)
print("✅ ANALYSIS COMPLETE! All 15 queries executed successfully.")
print("=" * 80)

E-COMMERCE DATA ANALYTICS PROJECT

1. BASIC STATISTICS
--------------------------------------------------
 total_orders  total_revenue  avg_order_value  total_units_sold  unique_customers  unique_products
         1200     1264761.96          1053.97              3535              1189                7

2. PRODUCT PERFORMANCE (Top 10)
--------------------------------------------------
Product  orders  units_sold   revenue  avg_order_value
  Chair     178         562 195620.11          1098.99
Printer     181         542 195612.61          1080.73
 Laptop     173         535 192126.56          1110.56
 Tablet     179         497 186568.95          1042.28
Monitor     163         480 175651.41          1077.62
   Desk     170         508 167459.93           985.06
  Phone     156         411 151722.39           972.58

3. ORDER STATUS DISTRIBUTION
--------------------------------------------------
OrderStatus  order_count  percentage  total_value
  Cancelled          250       20.83    2